# Computational Theory: SHA-256

This notebook works through the problems set out in the assessment brief, building up an implementation of SHA-256 as described in the [Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4).

## Problem 0: GitHub Issues

Progress on each problem below is tracked using GitHub Issues on this repository. See the [Issues tab](../../issues) for details.

## Problem 1: Representing SHA
Problem one introduces on how how SHA can be represented through various data structures within python


### 32-bit words 
FIPS 180-4 defines a word as a group of 32 or 64 bits depending on the SHA algorithim that is used ([FIPS 180-4, Section 2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=9)).

32 bit words can be represented via two distinct data structures in python.
1. Plain int: Which has arbitrary precision, meaning it can grow as large as needed but never overflows. It will happily grow into a 33 bit number. To prevent this, you must force it to behave like a 32-bit word by explicitly discarding any bits above the 32nd (& 0xFFFFFFFF) after every single addition.([Python docs: Numeric Types](https://docs.python.org/3/builtins/stdtypes.html#numeric-types-int-float-complex))
2. Numpy: Provides uint32. Which is a native, fixed-size 32-bit unsigned integer. It features automatic hardware-level overflow. If a value exceeds 0xFFFFFFFF, it natively wraps around to 0 without requiring a manual mask.([Numpy docs: uint32](https://numpy.org/devdocs/reference/arrays.scalars.html#numpy.uint32))

In conclusion, 32 bit words would be best represented usings numpy's 32 bit data structure as SHA-256 relies heavily on modulo $2^{32}$ addition. When a value exceeds 0xFFFFFFFF, the extra bits must disappear, and the number must wrap back around to zero.

In [1]:
# import numpy under its conventionbal alias np
import numpy as np 

In [2]:
# Plain Python int: unlimited precision, so it grows past 32 bits.
x = 0xFFFFFFFF
x_plus_one = x + 1

print(f"int:    {x:#010x} + 1 = {x_plus_one:#x}")

# Masking with & 0xFFFFFFFF keeps only the lowest 32 bits.
print(f"masked: {x_plus_one & 0xFFFFFFFF:#010x}")

int:    0xffffffff + 1 = 0x100000000
masked: 0x00000000


As we can see the result above has 9 hex digits, meaning it needs 33 bits, one more than a word can hold. SHA 256 requires addition modulo $2^{32}$ ([FIPS 180-4, Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). So with a plain `int` masking would have to be performed after each addition. The masked result, `0x00000000`, is what a true 32-bit word should give, so it's the value we expect `uint32` to produce.

In [3]:
# uint fixed size 32 bit
max_word = np.uint32(x)
# trying to overflow the 32 bit uint
y = max_word + np.uint32(1)

# print shows that its wrapped back to 0 
print(f"uint32: {max_word:#010x} + 1 = {y:#010x}")

uint32: 0xffffffff + 1 = 0x00000000


C:\Users\jhann\AppData\Local\Temp\ipykernel_31432\513019921.py:4: RuntimeWarning: overflow encountered in scalar add
  y = max_word + np.uint32(1)


The `uint32` addition seen above returns `0x00000000`. The same as the masked value from the plain `int`. No mask needed, the wrapping was done automatically. `uint32` is fixed at 32 bits, so arithmetic wraps modulo $2^{32}$, which is exactly what FIPS 180-4 requires ([Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). The `RuntimeWarning` is encountered becuase NumPy assumes the overflow is a mistake. In SHA-256 this wrap around is intended, so the warning here can be ignored ([NumPy docs: overflow errors](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors)). This is why SHA-256's 32-bit words will be represented as `numpy.uint32`.

### Sequence of 32-bit words 
Having chosen `numpy.uint32` to represent a single word in the subsection above, we now decide which data structure is best suited to holding a sequence of 32-bit words. Sequences of 32-bit words appear throughout SHA-256:

| Sequence | Length | FIPS 180-4 |
|---|---|---|
| Message block | 16 words | [Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19) |
| Message schedule | 64 words |  [Section 6.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=26) |
| Round constants | 64 words |  [Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16) |
| Hash value | 8 words |  [Section 5.3.3](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=23) |

Again we have two options for storing sequences of 32 bit words in python:
1. Python list: Lists are mutable sequences, meaning that they can be altered post creation using methods such as append and pop. This does not meet the requirments needed for SHA-256 as SHA-256's sequences are fixed at 16, 64 or 8 words. It also doesn't enforce what goes in this is at the discretion of the developer. This is a drawback as it means a value larger than 32 bits can be stored, which again doesnt meet the requirments of SHA-256 ([Python list docs](https://docs.python.org/3/builtins/stdtypes.html#typesseq-list)).
2. NumPy array with `dtype=np.uint32`: The NumPy array gives us fixed lentgh, which matches the SHA-256 requirment. `dtype=np.uint32` guarantees every element really is a 32-bit word ([NumPy docs: numpy.ndarray](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)). 

In [4]:
# demonstartes a list will accept a value too large for a 32-bit word.
words_list = [np.uint32(1), np.uint32(2)]
words_list[0] = 0x100000000 # 33 bit size value
print(words_list)

[4294967296, np.uint32(2)]


The demonstration above shows us that a list is not suitable as it accepts a value over 32 bits. Thus, meaning the value is no longer classed as a word and the modulo $2^{32}$ arithmetic SHA-256 depends on can no longer function ([Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

In [5]:
# demonstartes array with type uint32 is fixed size
words_array = np.array([1, 2], dtype=np.uint32)
try:
    words_array[0] = 0x100000000
except OverflowError as e:
    print(f"OverflowError: {e}")

OverflowError: Python int too large to convert to C long


The results above demonstrate that a NumPy array with `dtype=np.uint32` gives us our fixed element size requirement needed for SHA-256. We can see this as the `OverFlowError` was raised rather than storing a value requiring 33 bits, guaranteeing each element fits in 32 bits. This allows us to conclude that 32 bit words will be represented using NumPy arrays with `dtype=np.uint32`.

### Input messages
Input messages in FIPS 180-4 are defined by being a bit string, meaning they are an ordered sequence of 0 and 1 bits. The length of a message is denoted by ℓ, being the number of bits ([FIPS 180-4: Section 5.1.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18)). Due to this, whatever format the input message is in must be converted to bits. Python's core built-in types for manipulating binary data are `bytes` and `bytearray`. `bytes` objects are immutable sequences of integers in the range of 0 <= x < 256 ([Python Docs: bytes](https://docs.python.org/3/builtins/functions.html#func-bytes)). There is no native sequence-of-bits type within Python. The smallest individual unit we can use for representation is a byte rather than bit, where each byte holds 8 bits. Python's `str` type represents sequences of Unicode characters rather than binary data([Python Docs: The String type](https://docs.python.org/3/howto/unicode.html)). Therefore, before a `str` can be used as an input message for SHA-256, it must first be encoded into a sequence of bytes using a character encoding such as UTF-8. We can therefore consider `str`, after encoding, and `bytes` as two approaches when discussing how input messages are supplied and represented in our implementation.

In [6]:
# str measures its length in characters, not bytes.
text = "seán" # non ASCII accented character
# print length of text as represented in python 
print(f"str:   {text!r}  len = {len(text)} characters")
# print encoded version of text
print(f"bytes: {text.encode('utf-8')!r}  len = {len(text.encode('utf-8'))} bytes")

# str cannot be interpreted as binary data. byte is needed
try:
    np.frombuffer(text, dtype=np.uint32)
except TypeError as e:
    print(f"TypeError: {e}")

str:   'seán'  len = 4 characters
bytes: b'se\xc3\xa1n'  len = 5 bytes
TypeError: a bytes-like object is required, not 'str'


The demonstration above outlines the pitfalls with using `str`. We first print the length of `text` in its string format and also encoded format. We can deduce that an error would be made inferring ℓ from the string as for seán it shows to be 8 * 4 = 32 bits, while the encoded version would be 8 * 5 = 40 bits, this is the actual representation of its length. This underlines that if a non-ASCII character is present in the input message its string length will differ from its encoded version. When we try pass text as a parameter into the numpy `np.frombuffer` of type `dtype=np.uint32` we get a `TypeError` highlighting a bytes-like object is required, `str` is non-applicable.

In [7]:
# bytes is a sequence of integers, each one byte, so the bit length is exact.
message = text.encode("utf-8")
print(f"bytes:    {message!r}")
print(f"elements: {list(message)}")
print(f"length:   {len(message)} bytes = {len(message) * 8} bits")
print(f"as uint8: {np.frombuffer(message, dtype=np.uint8)}")

bytes:    b'se\xc3\xa1n'
elements: [115, 101, 195, 161, 110]
length:   5 bytes = 40 bits
as uint8: [115 101 195 161 110]


The cell above encodes `text` using UTF-8 and assigns the result to `message`. We print four things: the `bytes` literal, showing the raw byte values with the non-ASCII character represented as the escape sequences \xc3\xa1; the elements as a list, confirming that a bytes object is a sequence of integers in the range 0 to 255 as documented; the length in bytes alongside the length in bits, which is the value FIPS 180-4 denotes by ℓ; and the result of np.frombuffer, which now succeeds where it failed for str. Unlike a `str`, the length of a bytes object is unambiguous. Each element is exactly one byte, so ℓ is simply len(message) * 8, with no dependence on the characters the message happens to contain. The conversion into a NumPy array also succeeds without any encoding decision being required, as the data is already binary.Note that `dtype=np.uint8` is used here rather than np.uint32, as the message is a sequence of individual bytes at this stage. Grouping bytes into 32-bit words is only possible once padding has produced blocks of a fixed size, which is addressed in a later problem. We therefore represent input messages as bytes, encoding any str input to UTF-8 before it is hashed.

### 512-bit message blocks
According to Section 5.2.1 the message along with its padding is parsed into 512-bit blocks. It also outlines that this 512-bit block may be represented using sixteen 32-bit words ([FIPS 180-4: Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19)). 512 bits = 64 bytes = 16 words of 32 bits. We have already established 32-bit words are represented via `np.uint32` in a NumPy array. As a block is 16 such words, it follows that a block is represented via a NumPy array of 16 `uint32`. Byte order when representing message blocks will be big-endian as per FIPS 180-4 convention; this means that the left-most byte in the sequence is the most significant, little-endian being when the right-most byte is the most significant ([FIPS 180-4: Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=12)). This will also be addressed in the subsequent section.

In [8]:
# stand in for padded block, gives us sequence of increasing bytes
block = bytes(range(64))

# message block is 512-bits, or 64 bytes.
print(f"block:  {len(block)} bytes = {len(block) * 8} bits")

# parse the block into 32-bit words, read big-endian as FIPS 180-4 requires.
words = np.frombuffer(block, dtype=">u4")

# the block parses to exactly 16 words
print(f"words:  {len(words)} words, dtype = {words.dtype}")

# demonstrates the words as hex, so the grouping of four bytes into each word is visible.
print(f"hex:    {[f'{w:08x}' for w in words]}")

block:  64 bytes = 512 bits
words:  16 words, dtype = >u4
hex:    ['00010203', '04050607', '08090a0b', '0c0d0e0f', '10111213', '14151617', '18191a1b', '1c1d1e1f', '20212223', '24252627', '28292a2b', '2c2d2e2f', '30313233', '34353637', '38393a3b', '3c3d3e3f']


The output confirms the block structure set out in Section 5.2.1: 64 bytes yields exactly sixteen 32-bit words. The hex values show the big-endian convention from Section 3.1 in effect, with the first byte of each group occupying the most significant position. This allows us to conclude that 512-bit blocks will be represented as a NumPy array of sixteen `uint32` values, parsed from the blocks bytes in big-endian order. 

## Problem 2: SHA-256 Bitwise Operations

Discussion on the issue.

Aimed start date: Week begginning on 28th September

In [9]:
### Demo for prob 2 

## Problem 3: Generating the SHA-256 Constants

Discussion on the issue.

Aimed start date: Week begginning on 12th October

In [10]:
### Demo for prob 3 

## Problem 4: Padding and Parsing Messages

Discussion on the issue.

Aimed start date: Week beginning on 26th October.
Things to carry from Problem 1:
We obtained length as `len(message) * 8` as this translates bytes to bits. Taken from encoded bytes rather than the `str` as our input message. 
Padding must bring the message to a multiple of 512 bits so every block divides into sixteen 32-bit words.
FIPS 180-4's convention is the use of big-endian. Where the left-most byte is the most significant.
The parsing required here was demonstrated in Problem 1's 512-bit message block section, and is outlined in section 5.2.1 of FIPS 180-4.

In [11]:
### Demo for prob 4

## Problem 5: The SHA-256 Compression Function

Discussion on the issue.

Aimed start date: Week begginning on 9th November.

In [12]:
### Demo for prob 5 

## Problem 6: Complete SHA-256

Discussion on the issue.

Aimed start date: Week begginning on 23rd November.

In [13]:
### Demo for prob 6